# STEP 4-B — 최신 모델 비교

베이스라인으로 파이프라인이 살아있는 걸 확인했으니, 이제 좋은 모델을 찾습니다.

## 후보 (2026년 기준)

| 모델 | 계열 | 특징 |
|---|---|---|
| ResNet50 | 고전 CNN | 기준선 |
| EfficientNetV2-S | 효율 CNN | 가볍고 강함. 모바일 배포 1순위 |
| **ConvNeXt V2** | 현대 CNN | 질감 표현이 좋아 **피부에 잘 맞을 가능성** |
| Swin V2 | 계층적 ViT | 지역 패턴 + 전역 문맥 |
| EVA-02 | ViT (MIM) | 정확도 상한 확인용, 무거움 |
| SigLIP 2 | 이미지-텍스트 사전학습 | 소량 데이터에서 강한 편 |

> 💡 "가장 최신 = 가장 좋음"이 아닙니다. 데이터가 몇만 장 규모면
> 거대 모델은 오히려 과적합합니다. **실험으로 정합니다.**

## GPU 메모리 관리

Colab T4(16GB)에서 EVA-02 base 를 224px 로 돌리려면 배치를 줄이고
`grad_accum` 으로 실효 배치를 키워야 합니다. 아래 코드가 자동 처리합니다.

📖 [`docs/basics/09_ViT와_최신_백본_지도_2026.md`](../docs/basics/09_ViT와_최신_백본_지도_2026.md)

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    BASE = "/content" if os.path.isdir("/content") else (
           "/kaggle/working" if os.path.isdir("/kaggle/working") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "imagehash", "pyarrow", "grad-cam"], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

In [ ]:
from src import labels, split, data, models, train, evaluate
from src.config import MODEL_ZOO
import torch, gc

BEST_CROP = (env.work_root()/"best_crop.txt").read_text().strip() \
            if (env.work_root()/"best_crop.txt").exists() else "m1.5"
print("사용할 크롭:", BEST_CROP)

df = labels.load(env.work_root()/"manifests"/f"manifest_{BEST_CROP}.parquet")
tr, va = split.get_fold(df, 0)
print(f"train {len(tr):,} / val {len(va):,}")

In [ ]:
models.available()

## 1. 모델별 학습

시간이 오래 걸립니다. Colab 무료 티어는 세션이 끊길 수 있으니
**모델 하나씩 나눠 돌리고 결과를 저장**하세요. 아래 코드는 이미 끝난 모델을 건너뜁니다.

In [ ]:
def run_one(spec, epochs=15, effective_batch=64):
    """모델 하나를 학습하고 평가 결과를 돌려줍니다. VRAM 에 맞춰 배치/누적을 자동 조정."""
    name = spec.key
    ck = env.work_root()/"checkpoints"/f"zoo_{name}"/"best.pt"
    if ck.exists():
        print(f"⏭️  {name} — 이미 학습됨, 건너뜀")
        return None

    cfg = CFG(model_name=spec.timm_name, img_size=spec.img_size,
              epochs=epochs, exp_name=f"zoo_{name}")
    bs = env.suggest_batch_size(spec.img_size, spec.scale)
    cfg = CFG(**{**cfg.to_dict(), "batch_size": bs,
                 "grad_accum": max(1, effective_batch // max(bs, 1))})

    print(f"\n{'='*64}\n  {name}  |  {spec.img_size}px  |  "
          f"batch {bs} × accum {cfg.grad_accum} = 실효 {bs*cfg.grad_accum}\n{'='*64}")

    m = models.build(spec, len(CLASSES), pretrained=True,
                     drop_rate=cfg.drop_rate, drop_path_rate=cfg.drop_path_rate,
                     img_size=spec.img_size)
    ltr, lva, dtr, _ = data.build_loaders(tr, va, cfg, model=m)
    r = train.fit(m, ltr, lva, cfg, ds_train=dtr)
    _, lg, yy = train.evaluate_loader(m, lva, None, "cuda", len(CLASSES),
                                      tta_hflip=cfg.tta_hflip)
    out = evaluate.full_report(lg, yy, CLASSES, show=False)
    del m, ltr, lva; gc.collect(); torch.cuda.empty_cache()
    return out

In [ ]:
results = {}
for spec in MODEL_ZOO:
    try:
        r = run_one(spec)
        if r: results[spec.key] = r
    except torch.cuda.OutOfMemoryError:
        print(f"❌ {spec.key}: VRAM 부족 — img_size 를 줄이거나 건너뜁니다")
        gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f"❌ {spec.key}: {type(e).__name__}: {e}")

## 2. 결과 비교

In [ ]:
table = evaluate.compare_models(results)

### 표 읽는 법

- **macro_F1** 이 주 지표
- **CI_low ~ CI_high** 가 겹치는 모델끼리는 "더 낫다"고 말할 수 없습니다.
  0.78 vs 0.76 인데 CI 가 겹치면 그냥 우연입니다.
- **min_recall** (최악 클래스 재현율)이 낮으면 평균이 좋아도 위험합니다.
  A6(결절·종괴)를 계속 놓치는 모델은 못 씁니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ks = list(results); f1 = [results[k].metrics["macro_f1"] for k in ks]
lo = [results[k].ci[1] for k in ks]; hi = [results[k].ci[2] for k in ks]
order = np.argsort(f1)

fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(ks))
ax.barh(y, [f1[i] for i in order], color="#4C78A8")
ax.errorbar([f1[i] for i in order], y,
            xerr=[[f1[i]-lo[i] for i in order],[hi[i]-f1[i] for i in order]],
            fmt="none", ecolor="black", capsize=3)
ax.set_yticks(y, [ks[i] for i in order]); ax.set_xlabel("macro-F1 (95% CI)")
ax.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()
print("💡 오차막대가 겹치면 성능 차이가 통계적으로 유의하지 않습니다.")

## 3. 앙상블

**서로 다른 계열**(CNN + ViT)을 섞을 때 효과가 큽니다. 틀리는 방식이 다르기 때문입니다.
같은 모델을 seed 만 바꿔 섞으면 이득이 적습니다.

In [ ]:
from src.models import Ensemble, load_checkpoint
from src.config import MODEL_BY_KEY

TOP = list(table["model"].head(3))
print("앙상블 후보:", TOP)

member_models, tf_ref = [], None
for k in TOP:
    spec = MODEL_BY_KEY[k]
    m = load_checkpoint(str(env.work_root()/"checkpoints"/f"zoo_{k}"/"best.pt"),
                        spec, len(CLASSES))
    member_models.append(m)

ens = Ensemble(member_models).eval()

In [ ]:
# ⚠️ 앙상블 멤버들의 입력 해상도/정규화가 다르면 로더를 따로 써야 합니다.
#    여기서는 첫 멤버 기준으로 통일합니다 (간단하지만 약간 손해).
cfg_e = CFG(img_size=MODEL_BY_KEY[TOP[0]].img_size, exp_name="ensemble")
_, dl_e, _, _ = data.build_loaders(tr, va, cfg_e, model=member_models[0])

_, lg, yy = train.evaluate_loader(ens, dl_e, None, "cuda", len(CLASSES), tta_hflip=True)
results["ensemble"] = evaluate.full_report(lg, yy, CLASSES)

In [ ]:
evaluate.compare_models(results)

## 4. Mixup/CutMix 실험 (선택)

의료 이미지에서 Mixup 은 논쟁적입니다 — 존재하지 않는 병변 조합을 만들어내니까요.
실제로 도움이 되는지 직접 확인하세요.

In [ ]:
# best = table["model"].iloc[0]
# spec = MODEL_BY_KEY[best]
# cfg_mix = CFG(model_name=spec.timm_name, img_size=spec.img_size, epochs=15,
#               mixup_alpha=0.2, cutmix_alpha=0.2, exp_name=f"mixup_{best}")
# m = models.build(spec, len(CLASSES), pretrained=True)
# ltr, lva, dtr, _ = data.build_loaders(tr, va, cfg_mix, model=m)
# r = train.fit(m, ltr, lva, cfg_mix, ds_train=dtr)

## 5. 교차검증 (최종 성능 보고용)

fold 하나만 보면 운이 좋았을 수 있습니다. 5개 fold 평균 ± 표준편차가 정직한 숫자입니다.

⏱️ 시간이 5배 걸립니다. 최종 보고 직전에만 돌리세요.

In [ ]:
# BEST = table["model"].iloc[0]
# spec = MODEL_BY_KEY[BEST]
# fold_scores = []
# for k in range(CFG().n_folds):
#     t, v = split.get_fold(df, k)
#     cfg_k = CFG(model_name=spec.timm_name, img_size=spec.img_size,
#                 epochs=15, use_fold=k, exp_name=f"cv_{BEST}_f{k}")
#     m = models.build(spec, len(CLASSES), pretrained=True, verbose=False)
#     ltr, lva, dtr, _ = data.build_loaders(t, v, cfg_k, model=m)
#     r = train.fit(m, ltr, lva, cfg_k, ds_train=dtr, verbose=False)
#     fold_scores.append(r.best_score)
#     print(f"fold {k}: {r.best_score:.4f}")
#     del m; gc.collect(); torch.cuda.empty_cache()
# import numpy as np
# print(f"\n5-fold macro-F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

---
## ✅ 다음 단계

`05_평가_보정_GradCAM.ipynb`

거기서 **"이 모델을 실제로 써도 되는가"** 를 판단합니다.
정확도가 좋아도 배경을 보고 있으면 못 씁니다.

📖 함께 읽기:
- [`docs/basics/07_평가지표_의료AI_관점.md`](../docs/basics/07_평가지표_의료AI_관점.md)
- [`docs/basics/08_확률보정과_임계값_결정.md`](../docs/basics/08_확률보정과_임계값_결정.md)